# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [3]:

def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, banana, broccoli, or almonds."



In [1]:
get_food_calories('banana')

NameError: name 'get_food_calories' is not defined

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [11]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie and water-content information.
    You give concise answers. Use the calorie tool for calorie questions and the water-content
    tool for questions about how much water a food contains. Water values are approximate
    grams per 100 g of food.
    """,
    tools=[get_food_calories, get_food_water_content],
)

KeyboardInterrupt: 

In [17]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

Approximately 185 calories (banana ~105 kcal + apple ~80 kcal for medium sizes).


Enforce tools use:

In [12]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 185 calories total — 105 cal for a medium banana and 80 cal for a medium apple. Actual calories vary with size.


In [19]:
@function_tool
def get_food_water_content(food_item: str) -> str:
    """Get the approximate water content of a common food per 100 g."""
    water_content_per_100g = {
        "apple": 85.6,
        "banana": 74.9,
        "broccoli": 89.3,
        "almonds": 4.4,
    }

    food_key = food_item.strip().lower()
    water_grams = water_content_per_100g.get(food_key)
    if water_grams is None:
        return (
            f"I don't have water-content data for {food_item}. "
            "Try apple, banana, broccoli, or almonds."
        )
    return f"{food_item.title()}: approximately {water_grams} g of water per 100 g."

In [ ]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)